# 🌡️ Temperatura e Consumo de Energia — Uma Análise de Sensibilidade

## Contexto

Temperatura e consumo de energia são variáveis intimamente ligadas — mas a
relação entre elas é mais complexa do que parece. Não é apenas "mais calor,
mais energia". Depende da hora do dia, do dia da semana, da região do país
e de quanto a temperatura desvia do que é normal para aquele período.

Esta análise utiliza dados horários do **ONS (Operador Nacional do Sistema
Elétrico)** cruzados com dados meteorológicos do **Open-Meteo** para quatro
capitais representativas dos subsistemas brasileiros, cobrindo o período de
2019 a 2026.

## Perguntas que esta análise responde

**1. Como a temperatura se relaciona com a carga por região?**
O Brasil tem quatro subsistemas com climas completamente distintos. A
sensibilidade da carga à temperatura é igual no Nordeste e no Sul?

**2. Em quais horários a temperatura explica mais a carga?**
A correlação não é constante ao longo do dia — em alguns horários a
temperatura domina, em outros outros fatores competem. Onde está o
pico de sensibilidade?

**3. Qual é o lag temporal entre temperatura e carga?**
A carga responde à temperatura do momento atual, ou de horas atrás?
Uma onda de calor tem efeito imediato ou acumulado?

**4. Como a relação muda entre dia útil e fim de semana?**
Nos dias úteis a indústria amplifica o efeito da temperatura. No fim
de semana, o que sobra? A resposta surpreende.

**5. Em que ponto de temperatura a carga começa a reagir?**
Existe um threshold — abaixo dele a temperatura quase não importa,
acima dele a demanda por refrigeração explode. Onde está esse ponto
em cada região?

**6. Como dias mais quentes que o normal afetam a carga?**
Usando anomalia térmica mensal, isolamos o efeito do clima atípico
da sazonalidade normal — e medimos a elasticidade da carga a desvios
de temperatura em diferentes horários do dia.

---


## 1. Bibliotecas e constantes


In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import math
from statsmodels.nonparametric.smoothers_lowess import lowess


# ── CONSTANTES ────────────────────────────────────────────────────────
REGIOES_COORDS = {
    'SUDESTE':  {'lat': -23.55, 'lon': -46.63, 'cidade': 'São Paulo'},
    'NORDESTE': {'lat':  -3.73, 'lon': -38.52, 'cidade': 'Fortaleza'},
    'NORTE':    {'lat':  -3.10, 'lon': -60.02, 'cidade': 'Manaus'},
    'SUL':      {'lat': -30.03, 'lon': -51.23, 'cidade': 'Porto Alegre'},
}

CORES_REG = {
    'SUDESTE':  '#636EFA',
    'NORDESTE': '#EF553B',
    'NORTE':    '#00CC96',
    'SUL':      '#AB63FA',
}


## 2. Carregamento e Merge

In [ ]:
# ── Carregamento ──────────────────────────────────────────────────────
df_carga = pd.read_parquet('data/carga_diaria_ons_2019_2026.parquet')
df_temp  = pd.read_parquet('data/temperatura_openmeteo_2019_2026.parquet')

# ── Padronização de nomes ─────────────────────────────────────────────
df_carga['regiao'] = df_carga['nom_subsistema'].str.replace(
    'SUDESTE/CENTRO-OESTE', 'SUDESTE', regex=False)

# ── Merge por timestamp + região ──────────────────────────────────────
df = (df_carga
      .merge(
          df_temp[['time', 'temperature_2m', 'regiao']],
          left_on=['din_instante', 'regiao'],
          right_on=['time', 'regiao'],
          how='inner')
      .drop(columns=['time', 'nom_subsistema', 'id_subsistema'])
      .rename(columns={
          'din_instante':           'timestamp',
          'val_cargaenergiahomwmed': 'carga_mwmed',
          'temperature_2m':          'temperatura_c',
      }))

df['hora'] = df['timestamp'].dt.hour
df['mes']  = df['timestamp'].dt.month
df['data'] = df['timestamp'].dt.date

print(f"Shape: {df.shape}")
print(f"Período: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Nulos: {df.isna().sum().sum()}")
df.head()


## 3. Visão Geral — Temperatura × Carga

In [ ]:
# ── Scatter com trendline por região ─────────────────────────────────
fig = px.scatter(
    df.sample(30000, random_state=42),
    x='temperatura_c',
    y='carga_mwmed',
    color='regiao',
    color_discrete_map=CORES_REG,
    opacity=0.15,
    trendline='ols',
    facet_col='regiao',
    facet_col_wrap=2,
    title='<b>Temperatura × Carga por Região</b>',
    labels={
        'temperatura_c':  'Temperatura (°C)',
        'carga_mwmed':    'Carga (MWmed)',
        'regiao':         'Região'
    },
    template='plotly_white',
    height=600)

fig.update_traces(marker=dict(size=3))
fig.show()


![Gráfico de Dispersão de Temperatura](midia/SCATTER_TEMPERATURA.png)

### Observações

O scatter revela quatro perfis distintos.

O **Sudeste** tem a relação mais clara: dispersão ampla de temperatura com
tendência positiva evidente — cada grau a mais empurra a carga para cima.
O **Sul** segue padrão similar, mas com inclinação mais suave.

O **Nordeste** chama atenção pela concentração extrema — temperatura quase
sempre entre 25°C e 32°C, carga comprimida em uma faixa estreita. O clima
homogêneo ao longo do ano faz a temperatura explicar pouco da variação.

O **Norte** é o caso mais extremo: quase sem variação de temperatura e linha
de tendência horizontal. Aqui, temperatura sozinha não explica a dinâmica
de consumo.

O scatter tem limite como ferramenta — pontos sobrepostos escondem onde a
densidade real está. O hexbin resolve isso.

## 4. Densidade — Hexbin por Região

In [ ]:


# Mantendo o seu tema dark
sns.set_theme(style="dark", rc={
    "axes.facecolor":   "#121212",
    "figure.facecolor": "#121212"
})

regioes = list(CORES_REG.keys())
n_regioes = len(regioes)

n_cols = 2
n_rows = math.ceil(n_regioes / n_cols)

# Ajustada a altura por linha de 5 para 3.5 (gráfico mais compacto)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows), constrained_layout=True)
axes = axes.flatten()

for i, regiao in enumerate(regioes):
    ax = axes[i]
    
    df_reg = df[df["regiao"] == regiao].dropna(subset=["temperatura_c", "carga_mwmed"])
    
    hb = ax.hexbin(
        df_reg["temperatura_c"], df_reg["carga_mwmed"],
        gridsize=40, cmap="mako", bins="log", mincnt=1,
        linewidths=0.1, edgecolors="#222222"
    )
    
    # Encolhi um pouco mais a colorbar (shrink=0.6) para casar com a nova altura
    cb = fig.colorbar(hb, ax=ax, shrink=0.6, pad=0.02)
    cb.ax.yaxis.set_tick_params(colors="white", labelsize=8)
    
    ax.tick_params(colors="white", labelsize=8)
    ax.set_xlabel("Temperatura (°C)", fontsize=9, color="#cccccc", weight="bold")
    ax.set_ylabel("Carga (MWmed)",    fontsize=9, color="#cccccc", weight="bold")
    ax.set_title(f"Densidade Térmica — {regiao}", color="white", fontsize=11, weight="bold")
    
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%d°"))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))
    ax.grid(True, linestyle="--", alpha=0.1, color="white")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.show()


![Gráfico de Dispersão de Temperatura](midia/HEXBIN_TEMPERATURA.png)

### Observações

O hexbin confirma e aprofunda o que o scatter sugeria.

O **Sudeste** tem a nuvem mais oval e inclinada, com a  densidade concentrada entre
15°C e 25°C com carga entre 35k e 45k MWmed, mas com cauda clara se
estendendo para temperaturas altas. A relação positiva é evidente.

O **Nordeste** tem a nuvem mais circular e vertical, com a temperatura concentrada
numa faixa estreita (~26°C-30°C) enquanto a carga varia bastante no eixo Y.
Isso confirma que no Nordeste outros fatores explicam a variação de carga
muito mais do que a temperatura.

O **Norte** é o mais comprimido de todos, com a temperatura sem variação relevante
e densidade centralizada. O sistema opera num regime quase constante.

O **Sul** tem a nuvem mais assimétrica: densa no centro-baixo (frio + carga
moderada) com cauda se abrindo para cima à direita (calor + carga alta). É
a região com comportamento mais bimodal: inverno e verão produzem dois
regimes distintos.

A densidade confirma que a relação temperatura-carga varia por região.
Mas varia também ao longo do dia, em alguns horários a temperatura
domina, em outros praticamente desaparece. O próximo passo é quantificar
essa sensibilidade hora a hora.

## 5. Correlação por Hora do Dia

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ── Função de correlação por grupo ─────────────────────────────────────
def corr_pearson(group):
    if len(group) <= 10:
        return np.nan
    return group["temperatura_c"].corr(group["carga_mwmed"], method="pearson")

# ── Calcula correlação por (região, hora) ─────────────────────────────
df_corr = (
    df.groupby(["regiao", "hora"])
      .apply(corr_pearson)
      .reset_index(name="Pearson")
)

# ── Pivot ──────────────────────────────────────────────────────────────
pivot = df_corr.pivot(
    index="regiao",
    columns="hora",
    values="Pearson"
)

# ordem opcional
ordem_regioes = ["SUL", "SUDESTE", "NORTE", "NORDESTE"]
pivot = pivot.reindex(ordem_regioes)

# ── Texto apenas para correlações fortes ──────────────────────────────
text = np.where(
    pivot.values >= 0.60,
    np.round(pivot.values, 2),
    ""
)

# ── Heatmap ───────────────────────────────────────────────────────────
fig = go.Figure(
    go.Heatmap(
        z=pivot.values,
        x=list(range(24)),
        y=pivot.index.tolist(),
        text=text,
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorscale="Inferno",
        zmin=0,
        zmax=1,
        xgap=2,
        ygap=2,
        hoverongaps=False,
        colorbar=dict(
            title="Correlação",
            thickness=18
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Hora: %{x}h<br>"
            "Correlação: %{z:.3f}"
            "<extra></extra>"
        ),
        hoverlabel=dict(
            bgcolor="#222",
            font_size=13
        )
    )
)

# ── Layout ────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="<b>Sensibilidade da Carga à Temperatura (Análise Intradiária)</b>",
        x=0.5,
        font=dict(size=24)
    ),
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    height=500,
    margin=dict(l=90, r=40, t=90, b=70),
    font=dict(family="Arial", size=13)
)

# ── Eixos ─────────────────────────────────────────────────────────────
fig.update_xaxes(
    title_text="Hora do Dia",
    tickmode="linear",
    dtick=1,
    showgrid=False
)

fig.update_yaxes(
    title_text="Região",
    showgrid=False
)

fig.show()

![Gráfico de Dispersão de Temperatura](midia/HEATMAP_TEMPERATURA.png)

### Observações

O heatmap revela um padrão contraintuitivo: a correlação entre temperatura
e carga é máxima na madrugada, não no pico do calor.

No **Sudeste e Sul**, os coeficientes mais altos aparecem entre 0h e 4h
(0.63 a 0.67), exatamente quando o sistema está no seu nível mínimo de
atividade. De dia, quando o calor é maior, a correlação cai pela metade.

A explicação não é imediata. De madrugada, poucos fatores competem:
indústria parada, comércio fechado. A temperatura domina o sinal porque
o ruído é mínimo. Durante o dia, atividade industrial, comportamento
humano e calendário entram em cena e diluem o efeito térmico.

O **Sul** tem um buraco escuro às 18h-19h, com a correlação próxima de zero.
É exatamente o horário de pico de carga: as pessoas chegam em casa e
ligam tudo, mas o sol já caiu e a temperatura despencou. Carga alta com
temperatura baixa resulta em correlação zero ou negativa.

**Nordeste e Norte** permanecem uniformemente baixos — temperatura explica
pouco nessas regiões ao longo de todo o dia.


Esse padrão de correlação máxima justamente na madrugada nos leva a um conceito físico fundamental: a inércia térmica do ambiente construído. As paredes dos prédios, o asfalto e o concreto das metrópoles funcionam como grandes baterias de calor. Eles passam o dia acumulando radiação solar e demandam horas para transferir essa energia de volta para a atmosfera.Portanto, a temperatura das 3h da manhã não está isolada; ela carrega a herança térmica do dia anterior. O sistema elétrico, por consequência, sente o impacto desse calor sufocante que ficou retido nas estruturas (thermal lag), mantendo o uso de climatização ativo mesmo quando o sol já se foi. Para testar exatamente esse mecanismo de causa e efeito atrasado, a próxima etapa analisa a correlação da carga utilizando variáveis defasadas ($lag$) da temperatura.


## 6. Inércia Térmica e Análise de Defasagem

In [ ]:


# ── Garantir datetime ─────────────────────────────────────────────────
df["timestamp"] = pd.to_datetime(df["timestamp"])

# ── Configuração ──────────────────────────────────────────────────────
MAX_LAG = 24

resultados = []

# ── Loop por região ───────────────────────────────────────────────────
for regiao in df["regiao"].unique():

    sub = (
        df[df["regiao"] == regiao]
        .sort_values("timestamp")
        .copy()
    )

    # ── Testa cada lag ────────────────────────────────────────────────
    for lag in range(MAX_LAG + 1):

        temp_shift = sub["temperatura_c"].shift(lag)

        corr = temp_shift.corr(
            sub["carga_mwmed"],
            method="pearson"
        )

        resultados.append({
            "Regiao": regiao,
            "Lag": lag,
            "Pearson": corr
        })

# ── DataFrame final ───────────────────────────────────────────────────
df_lag = pd.DataFrame(resultados)

# ── Melhor lag por região ─────────────────────────────────────────────
idx = (
    df_lag
    .groupby("Regiao")["Pearson"]
    .idxmax()
)

df_best = df_lag.loc[idx]

print(df_best)

# ── Gráfico ───────────────────────────────────────────────────────────
fig = go.Figure()

for regiao in df_lag["Regiao"].unique():

    sub = df_lag[df_lag["Regiao"] == regiao]

    fig.add_trace(
        go.Scatter(
            x=sub["Lag"],
            y=sub["Pearson"],
            mode="lines+markers",
            name=regiao
        )
    )

# ── Layout ────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="<b>Correlação Temperatura × Carga por Lag</b>",
        x=0.5,
        font=dict(size=24)
    ),

    template="plotly_dark",

    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",

    height=600,

    xaxis=dict(
        title="Lag (horas)",
        dtick=1
    ),

    yaxis=dict(
        title="Correlação de Pearson"
    ),

    hovermode="x unified"
)

fig.show()

![Gráfico de Dispersão de Temperatura](midia/LAG_TEMPERATURA.png)

### Observações


No **Sudeste**, a correlação máxima ocorre com lag de 3 horas — a temperatura
de 3h atrás explica melhor a carga atual do que a temperatura do momento.
No **Sul**, o lag ótimo é de 2 horas. Nas outras regiões o sinal é fraco
demais para conclusão clara.

O mecanismo é intuitivo: uma onda de calor não gera demanda instantânea.
O ambiente precisa aquecer, as pessoas precisam ligar o ar-condicionado,
os equipamentos precisam responder. Esse processo leva horas.

Isso tem implicação direta para modelagem preditiva: incluir
`temperatura_lag_2h` e `temperatura_lag_3h` como features no modelo deve
melhorar a previsão de carga significativamente — especialmente no Sudeste
e no Sul.

Sabemos que a temperatura importa e que seu impacto muda dependendo da hora do dia. A próxima questão é estrutural: essa relação é linear em todos os horários ou existe um ponto de inflexão — onde o comportamento da carga muda? Para responder isso, a próxima análise investiga como a curva de carga reage a cada variação de temperatura durante a manhã, tarde, noite e madrugada

## 7. Pontos de Inflexão e Curvas de Carga por Horário

In [ ]:

# ── 1. ENGENHARIA DE FEATURES ─────────────────────────────────────────
df_refinado = df.copy()

def mapear_bloco(hora):
    if 0 <= hora < 6:   return "Madrugada (0h–6h)"
    elif 6 <= hora < 12: return "Manhã (6h–12h)"
    elif 12 <= hora < 18: return "Tarde (12h–18h)"
    else:                return "Noite (18h–0h)"

df_refinado["bloco_horario"] = df_refinado["hora"].apply(mapear_bloco)
df_refinado["mes"]            = df_refinado["timestamp"].dt.month

# Anomalia térmica: desvio em relação à média do mês/bloco/região
df_refinado["temp_anom"] = df_refinado["temperatura_c"] - df_refinado.groupby(
    ["regiao", "bloco_horario", "mes"])["temperatura_c"].transform("mean")

# Z-Score da carga por bloco/região
df_refinado["carga_norm"] = (
    df_refinado.groupby(["regiao", "bloco_horario"])["carga_mwmed"]
    .transform(lambda x: (x - x.mean()) / x.std()))

# ── 2. AGREGAÇÃO E FILTRO ─────────────────────────────────────────────
regioes = ["SUDESTE", "SUL"]
df_refinado = df_refinado[df_refinado["regiao"].isin(regioes)].copy()

df_plot = (df_refinado
           .groupby(["regiao", "bloco_horario", df_refinado["temp_anom"].round(1)])
           .agg({"carga_norm": "mean"})
           .reset_index())

df_counts = (df_refinado
             .groupby(["regiao", "bloco_horario", df_refinado["temp_anom"].round(1)])
             .size().reset_index(name="counts"))

df_plot = df_plot.merge(df_counts, on=["regiao", "bloco_horario", "temp_anom"])
df_plot = df_plot[df_plot["counts"] >= 20]

# ── 3. CONFIGURAÇÃO ───────────────────────────────────────────────────
BLOCOS_ORDEM = ["Madrugada (0h–6h)", "Manhã (6h–12h)",
                "Tarde (12h–18h)",   "Noite (18h–0h)"]

CORES_BLOCOS = {
    "Madrugada (0h–6h)": "#38bdf8",
    "Manhã (6h–12h)":    "#fbbf24",
    "Tarde (12h–18h)":   "#f43f5e",
    "Noite (18h–0h)":    "#a78bfa",
}

BG      = "#0d1117"
GRID    = "rgba(255,255,255,0.05)"
ZERO    = "rgba(255,255,255,0.15)"
TICK_C  = "#6b7280"
TITLE_C = "#f1f5f9"

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["SUDESTE", "SUL"],
    horizontal_spacing=0.10)

# ── 4. TRACES ─────────────────────────────────────────────────────────
for col_idx, regiao in enumerate(regioes, start=1):
    sub_reg = df_plot[df_plot["regiao"] == regiao]

    for bloco in BLOCOS_ORDEM:
        sub = sub_reg[sub_reg["bloco_horario"] == bloco].sort_values("temp_anom")
        if len(sub) < 5:
            continue

        sm = lowess(sub["carga_norm"], sub["temp_anom"], frac=0.4)

        fig.add_trace(go.Scatter(
            x=sm[:, 0], y=sm[:, 1],
            mode="lines",
            name=bloco,
            line=dict(width=3.5, color=CORES_BLOCOS[bloco]),
            legendgroup=bloco,
            showlegend=(col_idx == 1),
            hovertemplate=(
                f"<b>{regiao} — {bloco}</b><br>"
                "Anomalia: %{x:+.1f}°C<br>"
                "Desvio da Carga: %{y:+.2f} σ"
                "<extra></extra>")),
            row=1, col=col_idx)

# ── 5. LAYOUT ─────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="<b>Elasticidade da Carga à Anomalia Térmica</b>"
             "<br><sup style='color:#6b7280'>Desvio da carga (σ) em função de dias mais quentes ou frios que o normal, por bloco horário</sup>",
        x=0.04, y=0.97,
        font=dict(size=20, color=TITLE_C)),
    template="plotly_dark",
    paper_bgcolor=BG, plot_bgcolor=BG,
    height=560,
    margin=dict(l=70, r=40, t=110, b=70),
    font=dict(family="Inter, Arial, sans-serif", size=12, color=TICK_C),
    legend=dict(
        title=dict(text="Bloco Horário", font=dict(color=TICK_C, size=11)),
        orientation="h", yanchor="bottom", y=1.04,
        xanchor="left", x=0.0,
        font=dict(size=11, color="#94a3b8"),
        bgcolor="rgba(0,0,0,0)"),
    hoverlabel=dict(
        bgcolor="#1e293b", font_size=12,
        bordercolor="#334155"))

# Títulos dos subplots
for ann in fig["layout"]["annotations"]:
    ann.update(font=dict(size=15, color=TITLE_C), y=1.02)

# Eixos
axis_x = dict(
    title_text="Anomalia de Temperatura (°C vs média do mês)",
    title_font=dict(color=TICK_C, size=11),
    tickfont=dict(color=TICK_C),
    dtick=2, showgrid=True, gridcolor=GRID,
    zeroline=True, zerolinecolor=ZERO, zerolinewidth=1.5,
    showline=False)

axis_y = dict(
    title_text="Elasticidade da Carga (Z-Score)",
    title_font=dict(color=TICK_C, size=11),
    tickfont=dict(color=TICK_C),
    showgrid=True, gridcolor=GRID,
    zeroline=True, zerolinecolor=ZERO,
    showline=False)

fig.update_xaxes(**axis_x)
fig.update_yaxes(**axis_y)
fig.update_yaxes(title_text="", row=1, col=2)

fig.show()

![Gráfico de Dispersão de Temperatura](midia/RELAÇÃO_TEMPERATURA.png)

### Observações

Este gráfico é o mais sofisticado da análise — ele isola o efeito de dias
atipicamente quentes ou frios, removendo a sazonalidade normal.

No **Sudeste**, o comportamento é linear e consistente em todos os blocos:
qualquer desvio positivo de temperatura aumenta a carga, qualquer desvio
negativo a reduz. A resposta é proporcional e simétrica — não importa o
horário, o sistema reage da mesma forma a anomalias térmicas.

O **Sul** é fundamentalmente diferente. A tarde (12h–18h) tem inclinação
negativa: dias mais quentes que o normal durante a tarde **reduzem** a
carga. É a assinatura do aquecimento elétrico — quando a tarde fica mais
quente que o esperado, menos aquecimento é necessário e o consumo cai.
É o único subsistema onde calor anômalo pode diminuir a demanda.

O Sul também exibe saturação: acima de +4°C de anomalia, as curvas param
de subir ou invertem. Existe um teto para o quanto calor extra aumenta
a demanda — provavelmente quando a capacidade de refrigeração já está
no limite.

A conclusão central é que **a mesma anomalia térmica produz efeitos
opostos dependendo da região e do horário**. Modelos que tratam
temperatura como variável simétrica e linear estão errados — pelo menos
para o Sul do Brasil.

A elasticidade térmica varia por região e por horário. Mas existe mais uma
dimensão que modifica essa relação: o dia da semana. Quando a indústria
para no fim de semana, o comportamento da carga muda — e a temperatura
passa a ter um papel diferente.

## 8. Correlação por dia da semana

In [ ]:
# ── Correlação por dia da semana ──────────────────────────────────────
df['dia_semana'] = pd.to_datetime(df['timestamp']).dt.dayofweek
DIAS_PT = ['Segunda','Terça','Quarta','Quinta','Sexta','Sábado','Domingo']

resultados_dia = []
for regiao in df['regiao'].unique():
    for dia in range(7):
        sub = df[(df['regiao'] == regiao) & (df['dia_semana'] == dia)].dropna()
        if len(sub) > 10:
            r = sub['temperatura_c'].corr(sub['carga_mwmed'], method='pearson')
            resultados_dia.append({
                'Região':     regiao,
                'Dia':        DIAS_PT[dia],
                'Dia_idx':    dia,
                'Pearson':    r
            })

df_dia_corr = pd.DataFrame(resultados_dia).sort_values('Dia_idx')
pivot_dia = df_dia_corr.pivot(index='Região', columns='Dia', values='Pearson')[DIAS_PT]

fig = px.imshow(
    pivot_dia,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='<b>Correlação Temperatura × Carga por Dia da Semana</b>',
    labels=dict(x='Dia da Semana', y='Região', color='Pearson (r)'),
    text_auto='.2f',
    height=350,
    aspect='auto',
    template='plotly_dark')

fig.update_layout(paper_bgcolor='#101010')
fig.show()

![Gráfico de Dispersão de Temperatura](midia/TEMPERATURA_DIA.png)

### Observações — Correlação por Dia da Semana

O padrão encontrado contraria a intuição.

Nos dias úteis, **Sudeste e Sul** mantêm correlação alta e estável (0.54
a 0.63) — a indústria e o comércio amplificam o efeito da temperatura,
pois ar-condicionado industrial e comercial adiciona volume à resposta.

No fim de semana a correlação **cai**, não sobe. Sem a amplificação
industrial, o sinal térmico fica mais fraco — o consumo residencial puro
é menos sensível a temperatura do que o mix industrial-comercial-residencial
dos dias úteis.

O caso mais extremo é o **Nordeste no domingo: correlação negativa (-0.21)**.
Dias mais quentes no domingo reduzem o consumo. A hipótese mais provável
é comportamental — calor intenso no fim de semana leva as pessoas a
reduzirem atividade, saírem de casa ou descansarem, derrubando a demanda.

A implicação para modelagem é direta: `temperatura × dia_da_semana` é
uma feature composta mais informativa do que temperatura isolada.

## Conclusão

Esta análise revelou que a relação entre temperatura e consumo de energia
é muito mais complexa do que a intuição sugere.

A correlação não é máxima quando o calor é maior — é máxima de madrugada,
quando poucos fatores competem com a temperatura. De dia, atividade
industrial, comportamento humano e calendário diluem o sinal térmico.

A análise de lag confirmou que a temperatura atual explica menos a carga
do que a temperatura de 2 a 3 horas atrás — o sistema tem memória térmica.
Uma onda de calor não gera demanda instantânea: o ambiente aquece, as
pessoas reagem, os equipamentos respondem.

O Sul é o caso mais rico: é o único subsistema onde calor anômalo pode
reduzir a carga — quando dias mais quentes substituem o aquecimento
elétrico do inverno. A mesma anomalia térmica produz efeitos opostos
dependendo da região, do horário e do dia da semana.

Esses achados têm implicação direta para o próximo passo desta série:
a construção de um modelo preditivo de carga horária. Temperatura não
entra como variável simples — entra como `temperatura_lag_2h`,
`temperatura_lag_3h` e como interação com dia da semana e bloco horário.
O modelo precisa capturar assimetrias que uma regressão linear jamais
capturia.